In [ ]:
using Random
using Statistics
using Printf
using LinearAlgebra
using Plots
using Logging

function find_project_root(start::AbstractString=pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "System1D.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate project root from $start")
        dir = parent
    end
end

PROJECT_ROOT = find_project_root()

include(joinpath(PROJECT_ROOT, "Experiments", "common", "notebook_helpers.jl"))

NOTEBOOK_REL_DIR = joinpath("Experiments", "systems", "cosine_lattice_ring_1d", "gfmc", "notebooks")
PATHS = nb_paths(PROJECT_ROOT, NOTEBOOK_REL_DIR)
nb_include_formatting(PATHS.notebook_dir)

include(joinpath(PROJECT_ROOT, "src", "System1D.jl"))
using .System1D

default(; dpi=170)
nothing


## Model and GFMC Parameters

This notebook studies a single particle in a cosine lattice on a periodic cell `x in [0, L)` with
`L = M * a` and potential `V(x) = V0 * cos(2*pi*x/a)`.

The notebook compares two fixed-population GFMC runs that share the same Hamiltonian and initialization:
- An unguided run
- An importance-sampled run driven by a trial state `log |psi_T(x)| = lambda * cos(2*pi*x/a)`

Parameters used below:
- Number of periods `M = 5`
- Lattice period `a = 2.0`
- Cell length `L = 10.0`
- Lattice amplitude `V0 = -1.0`
- Trial parameter `lambda = -V0 / 2`
- Time step `dt = 3.0e-3`
- Total steps `nsteps = 2000`
- Equilibration steps `nequil = 300`
- Target population `targetN = 2000`
- Feedback strength `feedback = 0.2`
- Reconfiguration interval `reconfiguration_interval = 5`
- Branch-weight cap `branch_cap = 5.0`
- ET averaging window `energy_window = 50`

Trial / node structure:
- Guided run uses `ImportanceGuiding(trial, H)`
- Unguided run uses `NoGuiding()`
- Both runs use `NoNode()` and `SystematicReconfiguration()`


## Julia Construction

The next cell defines the periodic Hamiltonian, the trial wavefunction, the two GFMC parameter objects, and the shared output toggles.

Change `M`, `a`, `V0`, or `CSV_FILENAME` there if you want a different lattice cell or saved table name.


In [ ]:
M = 5
a = 2.0
L = M * a
V0 = -5.0
k_lat = 2pi / a
lambda_trial = -V0 / 2

bc = PeriodicBoundary1D(0.0, L)
V(R) = V0 * cos(k_lat * R[1])
H = Hamiltonian(1, 0.5, V, bc)

logpsi(R) = lambda_trial * cos(k_lat * R[1])
gradlogpsi(R) = [-lambda_trial * k_lat * sin(k_lat * R[1])]
lapllogpsi(R) = -lambda_trial * k_lat^2 * cos(k_lat * R[1])
trial = TrialWF(logpsi, gradlogpsi, lapllogpsi)
guiding = ImportanceGuiding(trial, H)

targetN = 2000
dt = 3.0e-3
nsteps = 2000
nequil = 300
ET0 = -0.2
feedback = 0.2
reconfiguration_interval = 5
branch_cap = 5.0
energy_window = 50

params_plain = GFMCParams(dt, nsteps, nequil, targetN, ET0, feedback, reconfiguration_interval, branch_cap, energy_window)
params_guided = GFMCParams(dt, nsteps, nequil, targetN, ET0, feedback, reconfiguration_interval, branch_cap, energy_window)
RECONFIGURATION = SystematicReconfiguration()

rng_init = MersenneTwister(1234)
initial_positions = [[rand(rng_init) * L] for _ in 1:targetN]

SNAPSHOT_STEPS = nb_default_snapshot_steps(nsteps)
NBINS = 160
DENSITY_SMOOTHING = 9
DENSITY_RANGE = (0.0, L)
X_AXIS_LABEL = "x"
PERIOD_MARKERS = collect(0.0:a:L)

PLOT_TITLE = "Cosine lattice GFMC"
DENSITY_TITLE = "Cosine lattice GFMC: final density comparison"
PLOT_MODE = :compare_final_density
RUN_LABELS = ["unguided", "guided"]
RUN_COLORS = [:firebrick, :teal]

SHOW_PROGRESS = false
PROGRESS_EVERY = 0
DEBUG_MODE = false
DEBUG_EVERY = 25
WRITE_RUN_CSV = false
CSV_FILENAME = "cosine_lattice_ring_gfmc.csv"
SAVE_FIGURES = false
FIGURE_STEM = "cosine_lattice_ring_gfmc"


In [ ]:
sim_plain = GFMCSim(
    H,
    params_plain,
    initial_positions,
    MersenneTwister(52);
    reconfiguration=RECONFIGURATION,
)
run_gfmc!(
    sim_plain;
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABELS[1],
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

sim_guided = GFMCSim(
    H,
    params_guided,
    initial_positions,
    MersenneTwister(77);
    guiding=guiding,
    reconfiguration=RECONFIGURATION,
)
run_gfmc!(
    sim_guided;
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABELS[2],
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

for (label, sim_ref) in zip(RUN_LABELS, (sim_plain, sim_guided))
    start_idx = min(params_plain.nequil + 1, length(sim_ref.energy_mean_history))
    mean_energy, sem_energy = nb_mean_sem(sim_ref.energy_mean_history[start_idx:end])
    println(@sprintf("%s mean energy after nequil=%d: %.8f +/- %.3e", label, params_plain.nequil, mean_energy, sem_energy))
    println(@sprintf("%s final fixed walker count = %d", label, sim_ref.population_history[end]))
    println(@sprintf("%s final mean weight = %.6f", label, sim_ref.mean_weight_history[end]))
    println(@sprintf("%s final effective population = %.2f", label, sim_ref.effective_population_history[end]))
end

if WRITE_RUN_CSV
    csv_path = joinpath(PATHS.tables_dir, CSV_FILENAME)
    rows = vcat(nb_gfmc_rows(RUN_LABELS[1], sim_plain), nb_gfmc_rows(RUN_LABELS[2], sim_guided))
    nb_write_csv(csv_path, rows)
    println("Wrote run CSV to: ", abspath(csv_path))
end

SIMS = [sim_plain, sim_guided]
SIM_LABELS = RUN_LABELS
SIM_COLORS = RUN_COLORS


In [ ]:
history_fig = nb_plot_gfmc_history(SIMS; labels=SIM_LABELS, colors=SIM_COLORS, title_prefix=PLOT_TITLE)
display(history_fig)
nb_save_figure(history_fig, PATHS.figures_dir, FIGURE_STEM, "history"; enabled=SAVE_FIGURES)

if DENSITY_RANGE === nothing
    coord_values = Float64[]
    for sim in SIMS
        append!(coord_values, nb_all_coordinates(sim; coord=1))
    end
    xlo, xhi = nb_padded_limits(coord_values; pad_frac=0.12)
else
    xlo, xhi = DENSITY_RANGE
end

density_fig = plot(
    xlabel=X_AXIS_LABEL,
    ylabel="density",
    title=DENSITY_TITLE,
    legend=:topright,
    xlims=(xlo, xhi),
)

if PLOT_MODE == :snapshot_density
    base_sim = SIMS[1]
    available_steps = SNAPSHOT_STEPS[1:min(length(SNAPSHOT_STEPS), length(base_sim.walker_positions_history))]
    for (snapshot, step_idx) in zip(base_sim.walker_positions_history, available_steps)
        centers, density = nb_density_curve_from_snapshot(
            snapshot;
            coord=1,
            nbins=NBINS,
            xmin=xlo,
            xmax=xhi,
            smoothing_window=DENSITY_SMOOTHING,
        )
        plot!(density_fig, centers, density; label="step $(step_idx)", color=SIM_COLORS[1], linewidth=2.2, alpha=0.85)
    end
else
    for (sim, label, color) in zip(SIMS, SIM_LABELS, SIM_COLORS)
        centers, density = nb_density_curve_from_snapshot(
            nb_last_snapshot(sim);
            coord=1,
            nbins=NBINS,
            xmin=xlo,
            xmax=xhi,
            smoothing_window=DENSITY_SMOOTHING,
        )
        plot!(density_fig, centers, density; label=label, color=color, linewidth=2.4)
    end
end

if PERIOD_MARKERS !== nothing
    for (k, xmark) in enumerate(PERIOD_MARKERS)
        vline!(density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "cell markers" : ""))
    end
end

display(density_fig)
nb_save_figure(density_fig, PATHS.figures_dir, FIGURE_STEM, "density"; enabled=SAVE_FIGURES)
